# Aula 02 — Introdução aos Dados Tabulares

**Aluno:** Davi Horácio de Souza · **Matrícula:** 20250059934

Neste exercício refaço o roteiro da aula 2: entender o que são dados tabulares, praticar Series e DataFrame do pandas num exemplo pequeno e aplicar uma rotina de inspeção inicial e de diagnóstico de qualidade a um dataset real, o **Online Retail**.

O trabalho continua no projeto de vendas criado na Aula 01, que é a própria pasta [`Unidade_01`](..): os dados ficam em `Unidade_01/data/`. Os comandos de terminal aparecem em blocos `bash` com a saída obtida, e o código pandas fica em células executáveis. Cada resultado é seguido de uma análise.

## 1. Retomando o projeto: suporte a arquivos Excel

```bash
cd ..
uv pip install openpyxl
uv pip freeze > requirements.txt
cat requirements.txt
```

Saída:

```text
Resolved 2 packages in 122ms
Installed 2 packages in 32ms
 + et-xmlfile==2.0.0
 + openpyxl==3.1.5
et-xmlfile==2.0.0
numpy==2.5.5
openpyxl==3.1.5
pandas==3.1.1
python-dateutil==4.0.0
tzdata==2026.6
```

**Análise:** o pandas lê arquivos Excel através do **openpyxl**, que não vem com ele. Por isso, antes de abrir o dataset, instalei o pacote no `.venv` do projeto (junto veio o `et-xmlfile`, uma dependência dele) e atualizei o `requirements.txt` com o `freeze`. A lista agora tem seis pacotes com versão fixa: quem recriar o ambiente já recebe o suporte a Excel, sem descobrir na hora do erro que faltava um pacote.

## 2. Pandas em escala pequena: Series e DataFrame

In [1]:
from pathlib import Path

import pandas as pd

In [2]:
vendas = pd.DataFrame(
    {
        "produto": ["Caneca", "Camiseta", "Caneca", "Adesivo", "Camiseta"],
        "quantidade": [2, 1, 4, 10, 3],
        "preco_unitario": [35.0, 79.9, 35.0, 5.5, 79.9],
        "pais": ["Brasil", "Brasil", "Portugal", "Brasil", "Argentina"],
    }
)
vendas

,produto,quantidade,preco_unitario,pais
0,Caneca,2,35.0,Brasil
1,Camiseta,1,79.9,Brasil
2,Caneca,4,35.0,Portugal
3,Adesivo,10,5.5,Brasil
4,Camiseta,3,79.9,Argentina


**Análise:** um DataFrame é uma tabela: cada **linha** é uma observação (uma venda) e cada **coluna** é uma variável (produto, quantidade, preço, país). O pandas numerou as linhas de 0 a 4 automaticamente: esse é o **índice**. Começar com cinco linhas inventadas deixa fácil conferir de cabeça se cada operação seguinte está certa.

In [3]:
vendas.dtypes

produto               str
quantidade          int64
preco_unitario    float64
pais                  str
dtype: object

**Análise:** cada coluna tem um único tipo. `quantidade` é inteiro (`int64`), `preco_unitario` é decimal (`float64`), e `produto` e `pais` são texto (`str`). Isso corresponde aos tipos de variável: as duas primeiras são **quantitativas** (discreta e contínua) e as duas últimas **categóricas**. O tipo define o que dá para fazer com a coluna: somar quantidades faz sentido, somar nomes de países não.

In [4]:
type(vendas["produto"])

pandas.Series

**Análise:** cada coluna de um DataFrame, isolada, é uma **Series**: um vetor de valores com índice. Um DataFrame é, na prática, um conjunto de Series que compartilham o mesmo índice. Por isso as operações de coluna da próxima célula funcionam linha a linha automaticamente.

In [5]:
vendas["faturamento"] = vendas["quantidade"] * vendas["preco_unitario"]
vendas

,produto,quantidade,preco_unitario,pais,faturamento
0,Caneca,2,35.0,Brasil,70.0
1,Camiseta,1,79.9,Brasil,79.9
2,Caneca,4,35.0,Portugal,140.0
3,Adesivo,10,5.5,Brasil,55.0
4,Camiseta,3,79.9,Argentina,239.7


**Análise:** a nova coluna `faturamento` foi calculada multiplicando duas Series de uma vez, sem laço `for`. O pandas alinha os valores pelo índice e faz a conta linha a linha (operação **vetorizada**). A venda de maior faturamento é a de camisetas para a Argentina (239,70), mesmo sendo só 3 unidades: o preço unitário pesa mais que a quantidade.

In [6]:
filtro_brasil = vendas["pais"].eq("Brasil")
vendas.loc[filtro_brasil, ["produto", "faturamento"]]

,produto,faturamento
0,Caneca,70.0
1,Camiseta,79.9
3,Adesivo,55.0


**Análise:** o filtro `pais == "Brasil"` gera uma Series de `True`/`False`, e o `.loc` mantém só as linhas `True`, já escolhendo as colunas que interessam. Sobram 3 das 5 vendas, que somam 204,90 de faturamento. Separar o filtro numa variável com nome (`filtro_brasil`) deixa o código legível e permite reaproveitá-lo.

In [7]:
filtro_faturamento_alto = vendas["faturamento"].gt(50)
vendas.loc[filtro_brasil & filtro_faturamento_alto]

,produto,quantidade,preco_unitario,pais,faturamento
0,Caneca,2,35.0,Brasil,70.0
1,Camiseta,1,79.9,Brasil,79.9
3,Adesivo,10,5.5,Brasil,55.0


**Análise:** combinar filtros com `&` exige que as duas condições sejam verdadeiras. O resultado é igual ao filtro anterior porque **todas** as vendas do Brasil já passam de 50 (a menor é o adesivo, com 55,00). Isso mostra que vale sempre conferir o resultado de um filtro: aqui o segundo critério não removeu nada.

In [8]:
vendas.describe().round(2)

,quantidade,preco_unitario,faturamento
count,5.00,5.00,5.00
mean,4.00,47.06,116.92
std,3.54,32.31,75.85
min,1.00,5.50,55.00
25%,2.00,35.00,70.00
50%,3.00,35.00,79.90
75%,4.00,79.90,140.00
max,10.00,79.90,239.70


**Análise:** o `describe()` resume só as colunas numéricas. A média de faturamento (116,92) fica bem acima da mediana (79,90) porque a venda da Argentina (239,70) puxa a média para cima, enquanto a mediana não se deixa levar por um valor extremo. Mesmo num exemplo de cinco linhas, a diferença entre média e mediana já indica uma distribuição assimétrica.

## 3. O dataset: Online Retail

```bash
curl -sSL -o data/online_retail.zip "https://archive.ics.uci.edu/static/public/352/online+retail.zip"
unzip -o data/online_retail.zip -d data/
mv "data/Online Retail.xlsx" data/online_retail.xlsx
ls -lh data/
```

Saída:

```text
Archive:  data/online_retail.zip
 extracting: data/Online Retail.xlsx  
total 46M
-rwx------ 1 davi davi 23M May 22  2023 online_retail.xlsx
-rw-rw-r-- 1 davi davi 23M Nov 22 12:25 online_retail.zip
```

**Análise:** o dataset foi baixado direto da fonte oficial (UCI) para a pasta `data/`. O `zip` tinha um único arquivo, `Online Retail.xlsx`, com espaço no nome, e por isso o renomeei para `online_retail.xlsx`: nomes sem espaço evitam aspas em todo comando e erros bobos de caminho. São 23 MB que **não** vão para o Git, porque `data/` está no `.gitignore`.

```bash
cat >> README.md <<'EOF'

## Dados

- **Dataset:** Online Retail (UCI Machine Learning Repository)
- **Fonte:** https://archive.ics.uci.edu/dataset/352/online+retail
- **Licença:** CC BY 4.0
- **Citação:** Chen, D. (2015). Online Retail [Dataset]. UCI Machine Learning Repository. https://doi.org/10.24432/C5BW33

Para baixar, rode na pasta `Unidade_01`:

    curl -L -o data/online_retail.zip "https://archive.ics.uci.edu/static/public/352/online+retail.zip"
    unzip -o data/online_retail.zip -d data/
    mv "data/Online Retail.xlsx" data/online_retail.xlsx
EOF
cat README.md
```

Saída:

```text
# Projeto Vendas
Analise desenvolvida na disciplina de Ciencia de Dados.

## Dados

- **Dataset:** Online Retail (UCI Machine Learning Repository)
- **Fonte:** https://archive.ics.uci.edu/dataset/352/online+retail
- **Licença:** CC BY 4.0
- **Citação:** Chen, D. (2015). Online Retail [Dataset]. UCI Machine Learning Repository. https://doi.org/10.24432/C5BW33

Para baixar, rode na pasta `Unidade_01`:

    curl -L -o data/online_retail.zip "https://archive.ics.uci.edu/static/public/352/online+retail.zip"
    unzip -o data/online_retail.zip -d data/
    mv "data/Online Retail.xlsx" data/online_retail.xlsx
```

**Análise:** como os dados não são versionados, o README do projeto passou a dizer de onde eles vêm, qual a licença, como citar e quais comandos os baixam. É isso que torna o projeto reproduzível: quem clonar o repositório encontra a pasta `data/` vazia, mas tem a receita para enchê-la.

## 4. O ritual de inspeção inicial

In [9]:
caminho_dados = Path("..") / "data" / "online_retail.xlsx"

nomes_colunas = {
    "InvoiceNo": "invoice_no",
    "StockCode": "stock_code",
    "Description": "description",
    "Quantity": "quantity",
    "InvoiceDate": "invoice_date",
    "UnitPrice": "unit_price",
    "CustomerID": "customer_id",
    "Country": "country",
}

retail = pd.read_excel(caminho_dados).rename(columns=nomes_colunas)
retail.head()

,invoice_no,stock_code,description,quantity,invoice_date,unit_price,customer_id,country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


**Análise:** o arquivo abriu e as colunas foram renomeadas para `snake_case` (minúsculas com `_`), um padrão mais fácil de digitar no código. As primeiras linhas mostram a estrutura: cada linha é **um item de uma nota fiscal**, e não uma nota inteira. A nota `536365` aparece em várias linhas, uma por produto, todas com o mesmo cliente, data e país.

In [10]:
retail.tail()

,invoice_no,stock_code,description,quantity,invoice_date,unit_price,customer_id,country
541904,581587,22613,PACK OF 20 SPACEBOY NAPKINS,12,2011-12-09 12:50:00,0.85,12680.0,France
541905,581587,22899,CHILDREN'S APRON DOLLY GIRL,6,2011-12-09 12:50:00,2.10,12680.0,France
541906,581587,23254,CHILDRENS CUTLERY DOLLY GIRL,4,2011-12-09 12:50:00,4.15,12680.0,France
541907,581587,23255,CHILDRENS CUTLERY CIRCUS PARADE,4,2011-12-09 12:50:00,4.15,12680.0,France
541908,581587,22138,BAKING SET 9 PIECE RETROSPOT,3,2011-12-09 12:50:00,4.95,12680.0,France


**Análise:** o final do arquivo confirma o período: a última nota é de 9 de dezembro de 2011, e a primeira (no `head`) de 1º de dezembro de 2010, ou seja, cerca de um ano de vendas. Olhar o fim também serve para ver se o arquivo não termina com linhas de total ou rodapé, comuns em planilhas, o que não acontece aqui.

In [11]:
retail.shape

(541909, 8)

**Análise:** são **541.909 linhas e 8 colunas**. É grande demais para olhar linha a linha, o que justifica uma rotina de inspeção com resumos em vez de rolar a tabela.

In [12]:
retail.info()

<class 'pandas.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column        Non-Null Count   Dtype         
---  ------        --------------   -----         
 0   invoice_no    541909 non-null  object        
 1   stock_code    541909 non-null  object        
 2   description   540455 non-null  object        
 3   quantity      541909 non-null  int64         
 4   invoice_date  541909 non-null  datetime64[us]
 5   unit_price    541909 non-null  float64       
 6   customer_id   406829 non-null  float64       
 7   country       541909 non-null  str           
dtypes: datetime64[us](1), float64(2), int64(1), object(3), str(1)
memory usage: 33.1+ MB


**Análise:** o `info()` mostra o tipo e quantos valores não nulos cada coluna tem. Dois pontos chamam atenção: `customer_id` é `float64`, embora seja um código inteiro, e tem só 406.829 valores preenchidos, ou seja, faltam valores. `invoice_no` é `object` (tipos misturados), porque a maioria das notas é número, mas algumas são texto começando com `C`. A data já veio como `datetime`, o que vai permitir análises ao longo do tempo.

In [13]:
retail.isna().sum().sort_values(ascending=False)

customer_id     135080
description       1454
stock_code           0
invoice_no           0
quantity             0
invoice_date         0
unit_price           0
country              0
dtype: int64

**Análise:** confirma os valores ausentes: **135.080 linhas sem cliente** (cerca de 25% do total) e **1.454 sem descrição**. As outras colunas estão completas. Um quarto das vendas sem cliente identificado limita qualquer análise por cliente, como frequência de compra ou ticket médio por cliente.

In [14]:
retail.describe()

,quantity,invoice_date,unit_price,customer_id
count,541909.000000,541909,541909.000000,406829.000000
mean,9.552250,2011-07-04 13:34:57.156386,4.611114,15287.690570
min,-80995.000000,2010-12-01 08:26:00,-11062.060000,12346.000000
25%,1.000000,2011-03-28 11:34:00,1.250000,13953.000000
50%,3.000000,2011-07-19 17:17:00,2.080000,15152.000000
75%,10.000000,2011-10-19 11:27:00,4.130000,16791.000000
max,80995.000000,2011-12-09 12:50:00,38970.000000,18287.000000
std,218.081158,NaN,96.759853,1713.600303


**Análise:** os extremos revelam problemas: `quantity` vai de **−80.995 a +80.995** e `unit_price` tem mínimo de **−11.062,06**. Quantidade negativa não é venda, e um valor exatamente igual com sinal oposto sugere um pedido gigante que foi cancelado. Já a mediana da quantidade é 3 e a do preço 2,08: a venda típica é pequena, e os extremos são exceções que distorceriam médias e somas se não forem tratadas.

In [15]:
retail["country"].value_counts().head(10)

country
United Kingdom    495478
Germany             9495
France              8557
EIRE                8196
Spain               2533
Netherlands         2371
Belgium             2069
Switzerland         2002
Portugal            1519
Australia           1259
Name: count, dtype: int64

**Análise:** o Reino Unido concentra **495.478 linhas (cerca de 91%)**, muito à frente da Alemanha (9.495) e da França (8.557). É uma loja britânica com clientes no exterior. Qualquer análise "por país" precisa levar isso em conta: um total geral reflete quase só o Reino Unido.

In [16]:
retail["country"].nunique()

38

**Análise:** são **38 países** distintos. Como o top 10 já cobre a grande maioria das linhas, a maior parte desses países tem pouquíssimas vendas.

## 5. Interrogando os dados: seleção e filtragem

In [17]:
filtro_alemanha = retail["country"].eq("Germany")
colunas_resumo = ["invoice_no", "description", "quantity", "unit_price"]

print(f"Linhas da Alemanha: {filtro_alemanha.sum()}")
retail.loc[filtro_alemanha, colunas_resumo].head()

Linhas da Alemanha: 9495


,invoice_no,description,quantity,unit_price
1109,536527,SET OF 6 T-LIGHTS SANTA,6,2.95
1110,536527,ROTATING SILVER ANGELS T-LIGHT HLDR,6,2.55
1111,536527,MULTI COLOUR SILVER T-LIGHT HOLDER,12,0.85
1112,536527,5 HOOK HANGER MAGIC TOADSTOOL,12,1.65
1113,536527,3 HOOK HANGER MAGIC GARDEN,12,1.95


**Análise:** a Alemanha, segundo maior mercado, tem 9.495 linhas. O filtro com `.eq("Germany")` e a seleção de colunas pelo `.loc` deixam a saída focada só no que interessa (nota, produto, quantidade e preço). As primeiras linhas mostram itens decorativos baratos (de 0,85 a 2,95) comprados em quantidades de 6 e 12, o que sugere clientes que compram para revenda.

In [18]:
filtro_caro_fora_uk = retail["unit_price"].gt(100) & retail["country"].ne("United Kingdom")

print(f"Itens acima de 100 fora do Reino Unido: {filtro_caro_fora_uk.sum()}")
retail.loc[filtro_caro_fora_uk, ["description", "unit_price", "country"]].head(10)

Itens acima de 100 fora do Reino Unido: 97


,description,unit_price,country
39777,CARRIAGE,150.00,France
45622,Manual,1126.00,Spain
54382,VINTAGE BLUE KITCHEN CABINET,125.00,Cyprus
54395,VINTAGE RED KITCHEN CABINET,125.00,Cyprus
70795,VINTAGE RED KITCHEN CABINET,125.00,Singapore
73446,Manual,1715.85,Spain
84117,RUSTIC SEVENTEEN DRAWER SIDEBOARD,145.00,Cyprus
84183,VINTAGE BLUE KITCHEN CABINET,125.00,Cyprus
84184,VINTAGE RED KITCHEN CABINET,125.00,Cyprus
96677,Manual,320.69,Cyprus


**Análise:** há 97 itens com preço acima de 100 fora do Reino Unido. Parte deles são móveis de verdade (armários vintage a 125), mas aparecem também descrições como `Manual` (com valores como 1.126,00 e 1.715,85) e `CARRIAGE` (frete). Esses não são produtos, e sim lançamentos manuais e cobranças de envio. O filtro por preço revelou que o dataset mistura vendas com outros tipos de registro, o que importa para não somá-los como se fossem vendas.

## 6. A caça aos problemas: diagnóstico de qualidade

In [19]:
quantidade_negativa = retail["quantity"].lt(0)
quantidade_negativa.sum()

np.int64(10624)

**Análise:** **10.624 linhas** têm quantidade negativa. Numa base de vendas, isso não pode ser uma venda comum: são devoluções, cancelamentos ou ajustes de estoque. Somar a coluna `quantity` sem tratar essas linhas daria um total errado.

In [20]:
cancelamento = retail["invoice_no"].astype(str).str.startswith("C")
(quantidade_negativa & cancelamento).sum()

np.int64(9288)

**Análise:** dessas 10.624, **9.288** são de notas que começam com `C`, o código de cancelamento do dataset. As outras **1.336** linhas negativas não têm o `C`, então são outro tipo de registro (provavelmente ajustes de estoque, como avarias). O diagnóstico mostra que "quantidade negativa" e "cancelamento" se sobrepõem, mas não são a mesma coisa.

In [21]:
preco_zero = retail["unit_price"].eq(0)
preco_zero.sum()

np.int64(2515)

**Análise:** **2.515 linhas** têm preço zero. Um item que sai de graça não é venda: pode ser brinde, ajuste ou erro de cadastro. Essas linhas também distorcem cálculos como o preço médio.

In [22]:
retail.duplicated().sum()

np.int64(5268)

**Análise:** **5.268 linhas** são cópias exatas de outra linha, coluna por coluna. Pode ser erro de registro ou o mesmo produto lançado duas vezes na mesma nota. Sem tratar, elas inflam contagens e faturamento. Antes de removê-las seria bom confirmar a causa, mas o diagnóstico já mostra que o problema existe.

## 7. Perguntas de negócio

### 7.1 Quantas transações registram o país como Brazil?

In [23]:
filtro_brazil = retail["country"].eq("Brazil")

print(f"Linhas: {filtro_brazil.sum()}")
print(f"Notas fiscais distintas: {retail.loc[filtro_brazil, 'invoice_no'].nunique()}")

Linhas: 32
Notas fiscais distintas: 1


**Análise:** o Brasil aparece em **32 linhas, mas todas de uma única nota fiscal**, ou seja, uma só compra com 32 produtos diferentes. Por isso a resposta depende da definição de "transação": são 32 itens, mas uma única transação. Contar linhas dá a impressão de um mercado maior do que é. Vale notar também que o dataset grafa o país em inglês (`Brazil`), e filtrar por `"Brasil"`, como no exemplo da seção 2, não encontraria nada.

### 7.2 Quantas linhas têm quantidade negativa e como elas são?

In [24]:
print(f"Linhas com quantidade negativa: {quantidade_negativa.sum()}")
retail.loc[quantidade_negativa, ["invoice_no", "description", "quantity", "unit_price", "customer_id"]].head()

Linhas com quantidade negativa: 10624


,invoice_no,description,quantity,unit_price,customer_id
141,C536379,Discount,-1,27.50,14527.0
154,C536383,SET OF 3 COLOURED FLYING DUCKS,-1,4.65,15311.0
235,C536391,PLASTERS IN TIN CIRCUS PARADE,-12,1.65,17548.0
236,C536391,PACK OF 12 PINK PAISLEY TISSUES,-24,0.29,17548.0
237,C536391,PACK OF 12 BLUE PAISLEY TISSUES,-24,0.29,17548.0


**Análise:** são **10.624 linhas** com quantidade negativa, e as primeiras confirmam o padrão do diagnóstico: notas com prefixo `C` (cancelamentos), quantidades pequenas (−1, −12, −24) e preços normais. Aparece até um `Discount`, que é um desconto lançado como item, não um produto. Essas linhas precisam ser separadas das vendas antes de qualquer cálculo de volume ou faturamento.

### 7.3 Existe linha com preço zero e cliente ausente ao mesmo tempo?

In [25]:
preco_zero_sem_cliente = preco_zero & retail["customer_id"].isna()

print(f"Linhas com preço zero e sem cliente: {preco_zero_sem_cliente.sum()}")
retail.loc[preco_zero_sem_cliente, ["invoice_no", "description", "quantity", "unit_price", "customer_id"]].head()

Linhas com preço zero e sem cliente: 2475


,invoice_no,description,quantity,unit_price,customer_id
622,536414,NaN,56,0.0,NaN
1970,536545,NaN,1,0.0,NaN
1971,536546,NaN,1,0.0,NaN
1972,536547,NaN,1,0.0,NaN
1987,536549,NaN,1,0.0,NaN


**Análise:** sim: **2.475 linhas** têm preço zero **e** cliente ausente ao mesmo tempo, quase todas as 2.515 com preço zero. Nos exemplos, a descrição também está vazia (`NaN`). Linhas sem preço, sem cliente e sem descrição não representam venda nenhuma: são quase certamente ajustes internos de estoque. Elas formam um grupo bem definido que pode ser removido na limpeza.

### 7.4 Pergunta própria: quais produtos a Alemanha mais compra?

In [26]:
vendas_alemanha = retail.loc[filtro_alemanha & ~cancelamento]
vendas_alemanha.groupby("description")["quantity"].sum().sort_values(ascending=False).head()

description
ROUND SNACK BOXES SET OF4 WOODLAND     1233
ASSORTED COLOURS SILK FAN              1164
POSTAGE                                1114
WOODLAND CHARLOTTE BAG                 1020
PACK OF 72 RETROSPOT CAKE CASES        1002
Name: quantity, dtype: int64

**Análise:** para responder sem contar devoluções, filtrei a Alemanha **excluindo as notas canceladas** e somei as quantidades por produto. O mais comprado é o `ROUND SNACK BOXES SET OF4 WOODLAND` (1.233 unidades). O terceiro lugar, `POSTAGE`, não é um produto: é a cobrança de frete, que aparece em quase todo pedido internacional. Isso reforça o diagnóstico da seção 5: antes de um ranking de produtos é preciso tirar da base os lançamentos que não são mercadoria.

## 8. Ficha de reconhecimento do dataset

| Item | Online Retail |
|---|---|
| **Fonte** | UCI Machine Learning Repository (id 352), licença CC BY 4.0 |
| **O que é** | Itens de notas fiscais de uma loja online britânica |
| **Período** | 01/12/2010 a 09/12/2011 |
| **Tamanho** | 541.909 linhas × 8 colunas |
| **Unidade de observação** | Uma linha = um produto dentro de uma nota fiscal |
| **Variáveis categóricas** | `invoice_no`, `stock_code`, `description`, `customer_id` (código, apesar de numérico), `country` (38 países, ~91% Reino Unido) |
| **Variáveis quantitativas** | `quantity` (discreta), `unit_price` (contínua) |
| **Data/hora** | `invoice_date` |
| **Valores ausentes** | `customer_id`: 135.080 (~25%), `description`: 1.454 |

**Três problemas de qualidade encontrados:**

1. **Quantidades negativas (10.624 linhas):** 9.288 são cancelamentos (nota com prefixo `C`) e 1.336 são outros ajustes.
2. **Preço zero (2.515 linhas):** 2.475 delas também não têm cliente, e a descrição costuma estar vazia, o que indica ajustes de estoque e não vendas.
3. **Linhas duplicadas (5.268):** cópias exatas que inflariam contagens e faturamento.

Também chamam atenção os **lançamentos que não são produtos** (`POSTAGE`, `CARRIAGE`, `Manual`, `Discount`) e os extremos de ±80.995 unidades. Tudo isso precisa ser tratado antes de qualquer análise de vendas, e é o ponto de partida para a próxima aula.

## Perguntas para consolidar

**1. Por que `customer_id` aparece como `float64`, se é um código inteiro?**

Porque a coluna tem valores ausentes. No pandas, o `NaN` (ausente) é um valor de ponto flutuante, e uma coluna `int64` comum não consegue guardá-lo. Basta um valor faltando para a coluna inteira ser convertida para `float64`. Por isso o tipo errado já é, por si só, um sinal de que há dados faltando, e o `isna()` confirmou: 135.080 linhas sem cliente.

**2. A documentação do dataset diz que não há valores ausentes, mas a inspeção mostra o contrário. Em quem confiar?**

Nos dados. A documentação descreve o que o arquivo *deveria* ter, mas só a inspeção mostra o que ele *tem*. O `info()` e o `isna()` mostraram 135.080 clientes e 1.454 descrições ausentes. Por isso a rotina de inspeção vem antes de qualquer análise: decisões tomadas com base na documentação, e não nos dados reais, levariam a resultados errados.

**3. Um colega clona o repositório e a pasta `data/` está vazia. Quais passos reproduzem o ambiente e os dados?**

O ambiente e os dados não estão no Git (estão no `.gitignore`), mas as instruções estão:

```bash
cd Unidade_01
uv venv
source .venv/bin/activate
uv pip install -r requirements.txt
```

Depois basta rodar os comandos da seção **Dados** do README do projeto (`curl`, `unzip` e `mv`) para baixar o Online Retail para a pasta `data/`.

## Conclusão

No exemplo pequeno pratiquei as peças básicas do pandas: DataFrame, Series, tipos de coluna, colunas calculadas, filtros e resumo estatístico. No Online Retail, a rotina de inspeção (`head`/`tail`, `shape`, `info`, `isna`, `describe` e contagem por país) mostrou em poucos comandos o tamanho, a estrutura e os problemas do dataset: um quarto das vendas sem cliente, devoluções e ajustes misturados às vendas, preços zerados, duplicatas e lançamentos que não são produtos. A principal lição é que **o dado bruto não está pronto para análise**: sem esse diagnóstico, um simples ranking de produtos colocaria o frete em terceiro lugar.